In [ ]:
import os

REPO_DIR = "/kaggle/working/Inter-SubNet"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RookieJunChen/Inter-SubNet.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q GPUtil mir_eval pesq pystoi tqdm toml colorful torch_complex 
!pip uninstall -y librosa
!pip install --no-cache-dir "librosa>=0.10,<1"


import librosa
print("VERSION:", librosa.__version__)
print("PATH:", librosa.__file__)

In [ ]:
metrics_path = "speech_enhance/audio_zen/metrics.py"
content = open(metrics_path).read()
content = content.replace("from pypesq import pesq as nb_pesq\n", "")
content = content.replace(
    "return nb_pesq(nb_ref, nb_est, 8000)",
    'return pesq(8000, nb_ref, nb_est, "nb")'
)
open(metrics_path, "w").write(content)

assert "pypesq" not in open(metrics_path).read(), "Patch chưa thành công!"
print("Đã patch metrics.py, không cần pypesq nữa")

In [ ]:
!wget -q -O /kaggle/working/InTerSubNet_EN.tar \
  https://github.com/hkha0801-sketch/PESEM-VS/raw/refs/heads/main/checkpoints/InTerSubNet_EN.tar

import os
print(os.path.getsize("/kaggle/working/InTerSubNet_EN.tar") / 1e6, "MB")  # phải ~26.3MB


In [ ]:
import os

DATA_ROOT = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH"
TRAIN_CLEAN = f"{DATA_ROOT}/TRAIN/CLEAN"
TRAIN_NOISE = f"{DATA_ROOT}/TRAIN/NOISE"
TEST_DIR    = f"{DATA_ROOT}/TEST"

WORK_DIR = "/kaggle/working/train_data_finetune"
os.makedirs(WORK_DIR, exist_ok=True)

!python -m speech_enhance.tools.gen_lst --dataset_dir "{TRAIN_CLEAN}" --output_lst "{WORK_DIR}/clean.txt"
!python -m speech_enhance.tools.gen_lst --dataset_dir "{TRAIN_NOISE}" --output_lst "{WORK_DIR}/noise.txt"

# không có RIR -> tạo file rỗng, và set reverb_proportion = 0 ở config sau
open(f"{WORK_DIR}/rir.txt", "w").close()

print("clean:", sum(1 for _ in open(f"{WORK_DIR}/clean.txt")))
print("noise:", sum(1 for _ in open(f"{WORK_DIR}/noise.txt")))


In [ ]:
import random, numpy as np, soundfile as sf, librosa
from pathlib import Path

random.seed(0)
VAL_DIR = Path("/kaggle/working/val_set/no_reverb")
(VAL_DIR / "noisy").mkdir(parents=True, exist_ok=True)
(VAL_DIR / "clean").mkdir(parents=True, exist_ok=True)

clean_files = [l.strip() for l in open(f"{WORK_DIR}/clean.txt")]
noise_files = [l.strip() for l in open(f"{WORK_DIR}/noise.txt")]

N_VAL = 30
SR = 16000
val_clean = random.sample(clean_files, min(N_VAL, len(clean_files)))

for i, cpath in enumerate(val_clean):
    clean_y, _ = librosa.load(cpath, sr=SR)
    npath = random.choice(noise_files)
    noise_y, _ = librosa.load(npath, sr=SR)

    # cắt/loop noise cho khớp độ dài clean
    if len(noise_y) < len(clean_y):
        reps = int(np.ceil(len(clean_y) / len(noise_y)))
        noise_y = np.tile(noise_y, reps)
    noise_y = noise_y[: len(clean_y)]

    # mix ở SNR ~5dB
    clean_rms = np.sqrt(np.mean(clean_y ** 2) + 1e-9)
    noise_rms = np.sqrt(np.mean(noise_y ** 2) + 1e-9)
    snr_db = 5
    scalar = clean_rms / (10 ** (snr_db / 20)) / (noise_rms + 1e-9)
    noisy_y = clean_y + noise_y * scalar

    peak = np.max(np.abs(noisy_y))
    if peak > 0.99:
        noisy_y = noisy_y / peak * 0.99
        clean_y = clean_y / peak * 0.99

    sf.write(VAL_DIR / "noisy" / f"mix_fileid_{i}.wav", noisy_y, SR)
    sf.write(VAL_DIR / "clean" / f"clean_fileid_{i}.wav", clean_y, SR)

print("Đã tạo", N_VAL, "cặp validation tại", VAL_DIR)


In [ ]:
config_content = f'''
[meta]
save_dir = "/kaggle/working/logs/Inter_SubNet_finetune"
description = "Fine-tune Inter-SubNet on custom Vietnamese noise-speech data"
seed = 0
port = "4396"
keep_reproducibility = false
use_amp = true


[acoustics]
n_fft = 512
win_length = 512
sr = 16000
hop_length = 256


[loss_function]
name = "mse_loss"
[loss_function.args]


[optimizer]
lr = 0.0001
beta1 = 0.9
beta2 = 0.999


[train_dataset]
path = "inter_subnet.dataset.dataset_train.Dataset"
[train_dataset.args]
clean_dataset = "{WORK_DIR}/clean.txt"
clean_dataset_limit = false
clean_dataset_offset = 0
noise_dataset = "{WORK_DIR}/noise.txt"
noise_dataset_limit = false
noise_dataset_offset = 0
num_workers = 4
pre_load_clean_dataset = false
pre_load_noise = false
pre_load_rir = false
reverb_proportion = 0.0
rir_dataset = "{WORK_DIR}/rir.txt"
rir_dataset_limit = false
rir_dataset_offset = 0
silence_length = 0.2
snr_range = [-5, 20]
sr = 16000
sub_sample_length = 3.072
target_dB_FS = -25
target_dB_FS_floating_value = 10


[train_dataset.dataloader]
batch_size = 6
num_workers = 4
drop_last = true
pin_memory = true


[validation_dataset]
path = "inter_subnet.dataset.dataset_validation.Dataset"
[validation_dataset.args]
dataset_dir_list = [
    "/kaggle/working/val_set"
]
sr = 16000


[model]
path = "inter_subnet.model.Inter_SubNet.Inter_SubNet"
[model.args]
sb_num_neighbors = 15
num_freqs = 257
look_ahead = 2
sequence_model = "LSTM"
sb_output_activate_function = false
sb_model_hidden_size = 384
weight_init = false
norm_type = "offline_laplace_norm"
num_groups_in_drop_band = 2
sbinter_middle_hidden_times = 0.8


[trainer]
path = "inter_subnet.trainer.trainer.New_Judge_Trainer"
[trainer.train]
clip_grad_norm_value = 10
epochs = 20
alpha = 1
save_checkpoint_interval = 1
[trainer.validation]
save_max_metric_score = true
validation_interval = 1
[trainer.visualization]
metrics = ["WB_PESQ", "NB_PESQ", "STOI", "SI_SDR"]
n_samples = 10
num_workers = 2
'''

with open("config/train.toml", "w") as f:
    f.write(config_content)

print("Đã ghi config/train.toml")


## Bước 6 — Chạy fine-tune

In [ ]:
trainer_path = "speech_enhance/audio_zen/trainer/base_trainer.py"
content = open(trainer_path).read()

old1 = 'torch.load(model_path.as_posix(), map_location="cpu")'
new1 = 'torch.load(model_path.as_posix(), map_location="cpu", weights_only=False)'

old2 = 'torch.load(latest_model_path.as_posix(), map_location="cpu")'
new2 = 'torch.load(latest_model_path.as_posix(), map_location="cpu", weights_only=False)'

if old1 in content:
    content = content.replace(old1, new1)
if old2 in content:
    content = content.replace(old2, new2)

open(trainer_path, "w").write(content)
print("Đã patch torch.load để dùng weights_only=False")

In [ ]:
!python -m speech_enhance.tools.train \
  -C config/train.toml \
  -N 1 \
  -P /kaggle/working/InTerSubNet_EN.tar

In [ ]:
BEST_CKPT = "/kaggle/working/logs/Inter_SubNet_finetune/train/checkpoints/best_model.tar"

!python -m speech_enhance.tools.inference \
  -C config/inference.toml \
  -M "{BEST_CKPT}" \
  -I "{TEST_DIR}" \
  -O /kaggle/working/enhanced_test
